# Subgraph type 2: different state (call it inside a node)

The subgraph has **its own state** (`text`, `translated`) with no keys in common with the parent.
So a normal parent node calls it with `invoke` and maps the values **in** and **out** itself.

```
Parent:    START -> answer_question -> translate (calls subgraph) -> END
Subgraph:          START -> translate_text -> check_translation -> END
```

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
# ---------- Subgraph: a reusable translator with its OWN state ----------
class TranslatorState(TypedDict):
    text: str
    language: str
    translated: str


def translate_text(state: TranslatorState):
    prompt = f"Translate to {state['language']}. Reply with only the translation.\n{state['text']}"
    return {'translated': model.invoke(prompt).content}


def check_translation(state: TranslatorState):
    prompt = (f"Improve this {state['language']} translation so it sounds natural. Reply with only the text.\n"
              f"Original: {state['text']}\nTranslation: {state['translated']}")
    return {'translated': model.invoke(prompt).content}


translator_builder = StateGraph(TranslatorState)
translator_builder.add_node('translate_text', translate_text)
translator_builder.add_node('check_translation', check_translation)
translator_builder.add_edge(START, 'translate_text')
translator_builder.add_edge('translate_text', 'check_translation')
translator_builder.add_edge('check_translation', END)

translator_subgraph = translator_builder.compile()

In [3]:
# ---------- Parent graph: completely different keys ----------
class QAState(TypedDict):
    question: str
    answer_english: str
    answer_hindi: str


def answer_question(state: QAState):
    return {'answer_english': model.invoke(f"Answer in 2 short sentences: {state['question']}").content}


def translate(state: QAState):
    # map parent -> subgraph state, run it, then map subgraph -> parent state
    result = translator_subgraph.invoke({'text': state['answer_english'], 'language': 'Hindi'})
    return {'answer_hindi': result['translated']}


parent_builder = StateGraph(QAState)
parent_builder.add_node('answer_question', answer_question)
parent_builder.add_node('translate', translate)  # a normal node that calls the subgraph
parent_builder.add_edge(START, 'answer_question')
parent_builder.add_edge('answer_question', 'translate')
parent_builder.add_edge('translate', END)

workflow = parent_builder.compile()

In [4]:
result = workflow.invoke({'question': 'Why is the sky blue?'})

print('ENGLISH:\n', result['answer_english'])
print('\nHINDI:\n', result['answer_hindi'])
print('\nKeys in parent result:', list(result))

ENGLISH:
 The sky appears blue because air molecules scatter shorter (blue) wavelengths of sunlight more efficiently than longer wavelengths. This Rayleigh scattering redirects blue light in all directions, filling the sky with a blue hue.

HINDI:
 आकाश नीला दिखाई देता है क्योंकि वायु के अणु कम तरंगदैर्घ्य वाले (नीले) सूर्य के प्रकाश को लंबी तरंगदैर्घ्य वाले प्रकाश से अधिक कुशलता से बिखेरते हैं। इस रेले बिखराव के कारण नीली रोशनी सभी दिशाओं में फैल जाती है, जिससे आकाश एक नीले रंग से भर जाता है।

Keys in parent result: ['question', 'answer_english', 'answer_hindi']


In [5]:
# The same subgraph is reusable on its own, with any language
print(translator_subgraph.invoke({'text': 'Good morning, how are you?', 'language': 'French'})['translated'])

Bonjour, comment allez‑vous?
